# NoFluffJobs API search

Query the NoFluffJobs posting search endpoint, inspect the response, and collect the returned postings into a clean list.

## 1. Imports

We use `httpx` for the HTTP request and `pandas` later for tabular display.

In [118]:
import httpx
import pandas as pd

## 2. Endpoint and query parameters

`params` are sent as query string arguments. Salary is requested in EUR per month. The `region`/`requirement` filters are left commented so the search stays broad.

In [119]:
url = "https://nofluffjobs.com/api/search/posting"

# Query parameters sent on the URL
params = {
    "salaryCurrency": "EUR",
    "salaryPeriod": "month",
    "region": "nl",          # 'nl', 'pl', 'cz', 'hu', etc.
    "language": "en",
    "page": 1,
}

## 3. Request body

The backend expects the key `criteriaSearch` (not `criteria`). Add filters such as `requirement` inside it to narrow the search.

In [120]:
payload = {
    "criteriaSearch": {
        # "requirement": ["python"]
    }
}

## 4. HTTP headers

A browser-like `User-Agent` plus `Referer`/`Origin` are required for the API to accept the request.

In [121]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Referer": "https://nofluffjobs.com/",
    "Origin": "https://nofluffjobs.com",
}

## 5. Send the request

In [122]:
# Query the first 10 pages
all_postings = []
for page in range(1, 20):
    params["page"] = page
    response = httpx.post(url, params=params, json=payload, headers=headers)
    print(f"page {page}: HTTP {response.status_code}")
    if response.status_code == 200:
        data = response.json()
        postings = data.get("postings", [])
        all_postings.extend(postings)
        print(f"  retrieved {len(postings)} postings")
    else:
        print(f"  failed: {response.text[:200]}")
print(f"Total postings collected: {len(all_postings)}")

page 1: HTTP 200
  retrieved 119 postings
page 2: HTTP 200
  retrieved 89 postings
page 3: HTTP 200
  retrieved 212 postings
page 4: HTTP 200
  retrieved 154 postings
page 5: HTTP 200
  retrieved 184 postings
page 6: HTTP 200
  retrieved 190 postings
page 7: HTTP 200
  retrieved 127 postings
page 8: HTTP 200
  retrieved 142 postings
page 9: HTTP 200
  retrieved 112 postings
page 10: HTTP 200
  retrieved 116 postings
page 11: HTTP 200
  retrieved 120 postings
page 12: HTTP 200
  retrieved 63 postings
page 13: HTTP 200
  retrieved 156 postings
page 14: HTTP 200
  retrieved 185 postings
page 15: HTTP 200
  retrieved 202 postings
page 16: HTTP 200
  retrieved 208 postings
page 17: HTTP 200
  retrieved 95 postings
page 18: HTTP 200
  retrieved 120 postings
page 19: HTTP 200
  retrieved 223 postings
Total postings collected: 2817


## 6. Parse and clean the response

On success, extract the `postings` list. Each posting is kept as-is for now; `salary` is read out to confirm it is present.

In [123]:
# Keep all retrieved postings
cleaned_postings = all_postings

## 7. Inspect the collected postings

In [124]:
# display(cleaned_postings)

### Optional: view as a DataFrame

Flatten a few key fields for easier inspection.

In [125]:
if cleaned_postings:
    df = pd.DataFrame([
        {
            "title": j.get("title"),
            "company": j.get("name"),
            "salary.from": j.get("salary", {}).get("from"),
            "salary.to": j.get("salary", {}).get("to"),
            "currency": j.get("salary", {}).get("currency"),
            "remote": j.get("fullyRemote"),
        }
        for j in cleaned_postings
    ])
    display(df)
else:
    print("No postings collected.")

,title,company,salary_from,salary_to,currency,remote
0,Lead ArcGIS DevOps Engineer,DCG sp. z o.o.,6384.0,7896.0,EUR,False
1,Lead ArcGIS DevOps Engineer,DCG sp. z o.o.,6384.0,7896.0,EUR,False
2,Lead ArcGIS DevOps Engineer,DCG sp. z o.o.,6384.0,7896.0,EUR,False
3,Lead ArcGIS DevOps Engineer,DCG sp. z o.o.,6384.0,7896.0,EUR,False
4,Lead ArcGIS DevOps Engineer,DCG sp. z o.o.,6384.0,7896.0,EUR,False
...,...,...,...,...,...,...
2812,Shopify Specialist,Univio,2184.0,4536.0,EUR,False
2813,Shopify Specialist,Univio,2184.0,4536.0,EUR,False
2814,Shopify Specialist,Univio,2184.0,4536.0,EUR,False
2815,Shopify Specialist,Univio,2184.0,4536.0,EUR,False


## 8. Persist to SQLite

Store each posting in a local SQLite database.

In [126]:
import sqlite3
import json as _json
import os

db_path = "job_postings.db"
# wipe old database so schema changes take effect
if os.path.exists(db_path):
    os.remove(db_path)
conn = sqlite3.connect(db_path)

# Flatten every field of each posting into columns, keeping all of them
df_all = pd.json_normalize(cleaned_postings)

# Make nested values (lists/dicts) storable in SQLite
df_all = df_all.map(lambda x: _json.dumps(x) if isinstance(x, (list, dict)) else x)

# Use a composite key to deduplicate regional variants of the same posting
df_all["pk"] = df_all["name"].astype(str) + "|" + df_all["title"].astype(str) + "|" + df_all["category"].astype(str)
df_all = df_all.set_index("pk")

# Create the table with the composite primary key
cols = ["pk"] + [c for c in df_all.columns]
col_defs = ", ".join(["pk TEXT PRIMARY KEY"] + [f'"{c}"' for c in df_all.columns])
conn.execute(f"CREATE TABLE nofluff ({col_defs})")

# Upsert: insert new rows, keep older records on conflict
placeholders = ", ".join(["?"] * len(cols))
col_names = ", ".join([f'"{c}"' for c in cols])
sql = f"INSERT OR IGNORE INTO nofluff ({col_names}) VALUES ({placeholders})"
for idx, row in df_all.iterrows():
    values = [idx] + [row[c] for c in df_all.columns]
    conn.execute(sql, values)
conn.commit()

cur = conn.cursor()
print(f"Saved postings; total in nofluff: {cur.execute('SELECT COUNT(*) FROM nofluff').fetchone()[0]}")
conn.close()


Saved postings; total in nofluff: 375
